# Bayesian Statistics

Notebook for **Chapter 9** of the *stats* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/stats/ch09.html

Tip: change a number, predict the output, then run the cell again.

## 9.1 The Bayesian Recipe: Prior, Likelihood, Posterior

## 9.2 Conjugate Priors: Beta-Binomial, Normal-Normal and Gamma-Poisson

## 9.3 Credible Intervals and the Bayesian A/B Test

## 9.4 When There Is No Shortcut: MCMC Intuition

In [ ]:
def logpost(th, k, n):                      # log of likelihood x flat prior
    if th <= 0 or th >= 1: return -np.inf   # outside 0..1 is impossible
    return k*np.log(th) + (n-k)*np.log(1-th)

def mh(k, n, step, N=20000, start=0.5, seed=0):
    r = np.random.default_rng(seed); th = start; lp = logpost(th, k, n)
    chain = np.empty(N)
    for i in range(N):
        prop = th + step * r.normal()       # propose a nearby value
        lq = logpost(prop, k, n)
        if np.log(r.uniform()) < lq - lp:   # accept with prob min(1, ratio)
            th, lp = prop, lq
        chain[i] = th                       # record (even if we stayed)
    return chain

In [ ]:
# not run here: PyMC is not installed in this environment
import pymc as pm
with pm.Model():
    pa = pm.Beta("pa", 1, 1); pb = pm.Beta("pb", 1, 1)
    pm.Binomial("ya", n=4000, p=pa, observed=480)
    pm.Binomial("yb", n=4000, p=pb, observed=532)
    pm.Deterministic("diff", pb - pa)
    idata = pm.sample(2000, chains=4)

## 9.5 Frequentist or Bayesian: Which to Use?

## 9.6 Checking a Bayesian Model: Prior and Posterior Predictive Checks

A posterior is only as good as the model that produced it. Bayes' rule will happily give you a tight, confident interval for the rate of a Poisson model even if the data are nothing like Poisson. The Bayesian workflow therefore has two checks that use the model as a data generator.

In [ ]:
import numpy as np
rng = np.random.default_rng(91)
# Observed: 120 days of support tickets. Truth is overdispersed (negative binomial), analyst assumes Poisson.
y = rng.negative_binomial(n=2, p=2/(2+5), size=120)          # mean 5, variance 17.5
print("observed: mean", round(y.mean(), 2), " variance", round(y.var(ddof=1), 2), " zeros", (y == 0).sum(), " max", y.max())
# Conjugate Poisson-Gamma: prior Gamma(2, 0.2) (mean 10), posterior Gamma(2 + sum y, 0.2 + n)
a, b = 2 + y.sum(), 0.2 + len(y)
lam = rng.gamma(a, 1 / b, 4000)
print(f"posterior mean rate {a/b:.2f}, 95% interval ({np.percentile(lam, 2.5):.2f}, {np.percentile(lam, 97.5):.2f})")
# Posterior predictive check: simulate a replicated dataset for each draw, compare statistics
rep = rng.poisson(lam[:, None], (4000, len(y)))
for name, f, obs in [("variance", lambda r: r.var(axis=1, ddof=1), y.var(ddof=1)),
                     ("max", lambda r: r.max(axis=1), y.max()),
                     ("zeros", lambda r: (r == 0).sum(axis=1), (y == 0).sum())]:
    t = f(rep)
    print(f"{name:>8}: observed {obs:6.2f}   replicated 95% range ({np.percentile(t, 2.5):.1f}, {np.percentile(t, 97.5):.1f})   P(T_rep >= T_obs) = {(t >= obs).mean():.3f}")
# Prior predictive check of a careless prior: log(rate) ~ Normal(0, 10^2)
lr = rng.normal(0, 10, 100000)
cnt = rng.poisson(np.exp(np.clip(lr, -30, 30)))
print("careless prior, share of prior predictive counts above 1,000,000:", round((cnt > 1e6).mean(), 3),
      "  share equal to 0:", round((cnt == 0).mean(), 3))

observed: mean 5.17  variance 20.48  zeros 10  max 21
posterior mean rate 5.17, 95% interval (4.78, 5.60)
variance: observed  20.48   replicated 95% range (3.9, 6.7)   P(T_rep >= T_obs) = 0.000
     max: observed  21.00   replicated 95% range (10.0, 15.0)   P(T_rep >= T_obs) = 0.000
   zeros: observed  10.00   replicated 95% range (0.0, 3.0)   P(T_rep >= T_obs) = 0.000
careless prior, share of prior predictive counts above 1,000,000: 0.083   share equal to 0: 0.476


## 9.7 Bayes Factors: Comparing Two Models

Credible intervals describe a parameter inside one model. Sometimes the question is which of two models the data favour: "is this coin fair (θ = 0.5) or biased (θ unknown)?" The **Bayes factor** answers it:

In [ ]:
import numpy as np
from scipy.special import betaln
from scipy.stats import binomtest
def log_bf10(k, n, a=1.0, b=1.0):
    # H0: theta = 0.5.  H1: theta ~ Beta(a, b).  Binomial coefficient cancels.
    log_m1 = betaln(k + a, n - k + b) - betaln(a, b)
    log_m0 = n * np.log(0.5)
    return log_m1 - log_m0
print("data           p-value   BF10 (Beta(1,1))  BF10 (Beta(10,10))  P(H1|data) at 50/50 prior odds")
for k, n in [(62, 100), (55, 100), (530, 1000), (5100, 10000), (5250, 10000)]:
    p = binomtest(k, n, 0.5).pvalue
    b1, b2 = np.exp(log_bf10(k, n)), np.exp(log_bf10(k, n, 10, 10))
    print(f"{k:>5}/{n:<6}  {p:8.1e}   {b1:15.3f}   {b2:17.3f}   {b1/(1+b1):12.3f}")

data           p-value   BF10 (Beta(1,1))  BF10 (Beta(10,10))  P(H1|data) at 50/50 prior odds
   62/100      2.1e-02             2.214               4.620          0.689
   55/100      3.7e-01             0.204               0.615          0.170
  530/1000     6.2e-02             0.239               0.810          0.193
 5100/10000    4.7e-02             0.093               0.325          0.085
 5250/10000    6.0e-07          3376.207           11622.591          1.000


## 9.8 Hierarchical Models: Partial Pooling

Suppose you run 8 stores and want each store's conversion rate. Two obvious answers are both flawed. **No pooling** uses each store's own rate. It is unbiased, but a store with 50 visitors and 0 sales gets a rate of exactly 0.000. **Complete pooling** uses one rate for everyone. It is stable but ignores real differences. A **hierarchical model** sits between them: each store has its own rate, and those rates are drawn from a shared population distribution, here Beta(α, β). The data estimate α and β, which decide how much to trust each store.

In [ ]:
import numpy as np
from scipy.optimize import minimize
from scipy.special import betaln
n = np.array([20, 35, 50, 80, 120, 200, 400, 800])
rng = np.random.default_rng(92)
true_p = rng.beta(4, 28, 8)                  # stores vary around 12.5 percent
k = rng.binomial(n, true_p)
raw = k / n
def nll(t):
    a, b = np.exp(t)
    return -(betaln(k + a, n - k + b) - betaln(a, b)).sum()     # beta-binomial marginal likelihood (constants dropped)
r = minimize(nll, [np.log(10), np.log(80)], method="Nelder-Mead")
a, b = np.exp(r.x)
pooled = k.sum() / n.sum()
partial = (k + a) / (n + a + b)
print(f"fitted population prior: Beta({a:.1f}, {b:.1f}), mean {a/(a+b):.3f}, equivalent prior sample size {a+b:.0f}")
print(f"complete pooling rate: {pooled:.3f}")
print("store   n     k   no pooling   partial pooling   weight on own data   true p")
for i in range(8):
    w = n[i] / (n[i] + a + b)
    print(f"{i+1:>5} {n[i]:>4} {k[i]:>5}   {raw[i]:10.3f}   {partial[i]:15.3f}   {w:18.2f}   {true_p[i]:6.3f}")
print("squared error vs truth: no pooling", round(((raw - true_p)**2).sum(), 5), " partial", round(((partial - true_p)**2).sum(), 5), " complete", round(((pooled - true_p)**2).sum(), 5))

fitted population prior: Beta(19.0, 190.9), mean 0.091, equivalent prior sample size 210
complete pooling rate: 0.089
store   n     k   no pooling   partial pooling   weight on own data   true p
    1   20     1        0.050             0.087                 0.09    0.032
    2   35     1        0.029             0.082                 0.14    0.109
    3   50     0        0.000             0.073                 0.19    0.111
    4   80    11        0.138             0.104                 0.28    0.194
    5  120    12        0.100             0.094                 0.36    0.096
    6  200    24        0.120             0.105                 0.49    0.086
    7  400    44        0.110             0.103                 0.66    0.116
    8  800    59        0.074             0.077                 0.79    0.072
squared error vs truth: no pooling 0.02355  partial 0.014  complete 0.01629


## 9.9 MCMC Diagnostics: R-hat, ESS and Divergences

Section 9.4 showed that a bad sampler can look fine. Three numbers tell you whether to trust a run. Always look at all three before you read a posterior.

In [ ]:
import numpy as np
def metropolis(logp, start, step, N, seed):
    r = np.random.default_rng(seed); x = start; lp = logp(x); out = np.empty(N); acc = 0
    for i in range(N):
        p = x + step * r.normal(); lq = logp(p)
        if np.log(r.uniform()) < lq - lp: x, lp = p, lq; acc += 1
        out[i] = x
    return out, acc / N
def split_rhat(ch):                       # ch: (chains, draws); split each chain in two
    m, n = ch.shape; h = n // 2
    s = np.vstack([ch[:, :h], ch[:, h:2*h]])
    W = s.var(axis=1, ddof=1).mean(); B = h * s.mean(axis=1).var(ddof=1)
    return np.sqrt(((h - 1) / h * W + B / h) / W)
def ess(ch):                              # between-chain-aware ESS (Vehtari et al. style, simple truncation)
    m, n = ch.shape; x = ch - ch.mean(axis=1, keepdims=True)
    W = ch.var(axis=1, ddof=1).mean(); B = n * ch.mean(axis=1).var(ddof=1)
    var_plus = (n - 1) / n * W + B / n
    acov = lambda l: np.mean([np.dot(c[:n-l], c[l:]) / n for c in x])
    rho = [1 - (W - acov(l)) / var_plus for l in range(300)]
    tau = 1.0
    for l in range(1, 299, 2):
        pair = rho[l] + rho[l + 1]
        if pair < 0: break
        tau += 2 * pair
    return m * n / tau
mix = lambda x: np.logaddexp(-0.5 * (x - 4) ** 2, -0.5 * (x + 4) ** 2)     # two bumps at -4 and +4
starts = [-4.0, -4.0, 4.0, 4.0]
res = {}
for name, step in [("step 0.5 (stuck)", 0.5), ("step 6.0 (mixes)", 6.0)]:
    chains, accs = zip(*[metropolis(mix, s, step, 4000, i) for i, s in enumerate(starts)])
    res[name] = np.array(chains)
    print(f"{name:>18}: acceptance {np.mean(accs):.2f}, split R-hat {split_rhat(res[name]):.2f}, ESS {ess(res[name]):.0f} of {res[name].size}, share of draws above 0: {(res[name] > 0).mean():.2f}")
print("truth: the target is symmetric, so the share above 0 should be 0.50 and the mean 0")

  step 0.5 (stuck): acceptance 0.84, split R-hat 4.44, ESS 28 of 16000, share of draws above 0: 0.50
  step 6.0 (mixes): acceptance 0.29, split R-hat 1.00, ESS 1317 of 16000, share of draws above 0: 0.52
truth: the target is symmetric, so the share above 0 should be 0.50 and the mean 0


## Chapter recap

## Practice set

**1 (easy).** Prior Beta(1, 1), data 8 successes in 20 trials. Posterior and mean? **Answer:** Beta(9, 13), mean 9/22 = 0.409.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/stats/ch09.html